# The settled numbers

Three corrections are folded in. Literal newlines inside JSON strings are escaped
before parsing. Records whose tool schemas do not extract are excluded and counted
rather than scored. And a `Finish` call whose input fails to parse is checked for the
presence of its required keys by pattern before any violation is declared, because
adjudication showed that 78 of 127 unparseable inputs were `Finish` calls carrying
`return_type`, truncated by the corpus rather than malformed by the agent.

Everything the manuscript might quote is recomputed here in one run, against the same
seed and the same 20,000 records, so that no figure comes from a different pass.

In [ ]:
import ast, itertools, json, math, re, time
from collections import Counter
from datasets import load_dataset

N_TEXT, N_EMB, SEED = 20000, 2000, 20261018
ds = load_dataset("tuandunghcmut/toolbench-v1", split="train", streaming=True)
rows = list(itertools.islice(iter(ds.shuffle(seed=SEED, buffer_size=10000)), N_TEXT))
print("records: %d, seed %d" % (len(rows), SEED))

APIS = re.compile(r"you have access to the following APIs:\s*(\[.*)", re.S)
STEP = re.compile(r"Action:\s*(?P<action>[^\n]*)\n+Action Input:\s*(?P<inp>.*)", re.S)

def turns(r):
    c = r["conversations"]
    return list(zip(c["from"], c["value"]))

def schemas(r):
    for role, val in turns(r):
        if role != "system":
            continue
        m = APIS.search(val)
        if not m:
            return None
        blob = m.group(1).strip()
        depth = 0
        for i, ch in enumerate(blob):
            depth += (ch == "[") - (ch == "]")
            if depth == 0:
                try:
                    return ast.literal_eval(blob[:i + 1])
                except Exception:
                    return None
    return None

def repair(raw):
    out, inside, esc = [], False, False
    for ch in raw:
        if esc:
            out.append(ch); esc = False; continue
        if ch == "\\":
            out.append(ch); esc = True; continue
        if ch == '"':
            inside = not inside
        if inside and ch in "\n\t\r":
            out.append({"\n": "\\n", "\t": "\\t", "\r": "\\r"}[ch]); continue
        out.append(ch)
    return "".join(out)

def parse_input(raw):
    raw = raw.strip()
    for cand in (raw, repair(raw)):
        for loader in (json.loads, ast.literal_eval):
            try:
                v = loader(cand)
                if isinstance(v, dict):
                    return v, "object"
            except Exception:
                pass
    return None, "unparseable"

def key_present(raw, key):
    """Repli : la cle requise figure-t-elle, meme si l'objet ne se parse pas."""
    return re.search(r'["\']%s["\']\s*:' % re.escape(key), raw) is not None

def steps(r):
    out, pending = [], None
    for role, val in turns(r):
        if role == "assistant":
            m = STEP.search(val)
            if not m:
                continue
            pending = {"thought": val.split("Action:")[0].replace("Thought:", "").strip(),
                       "action": m.group("action").strip(),
                       "input_raw": m.group("inp").strip(), "observation": None}
            out.append(pending)
        elif role == "function" and pending is not None:
            pending["observation"] = val
            pending = None
    return out

In [ ]:
def schema_flag(step, by):
    """Les deux controles que la Section IV.B annonce comme transferables."""
    a, raw = step["action"], step["input_raw"]
    if a not in by:
        return "tool not declared"
    req = (by[a].get("parameters", {}) or {}).get("required", []) or []
    val, how = parse_input(raw)
    if how != "object":
        # Le repli ne vaut que pour Finish, ou l'arbitrage a montre que l'echec vient
        # d'une troncature du corpus : 78 des 127 entrees non analysables etaient des
        # Finish portant bien leur return_type. Ailleurs, une entree non analysable est
        # une faute de l'agent dans 43 cas sur 49 examines, valeurs JavaScript ou noms
        # de variables non resolus, et la clef peut etre presente alors que la valeur
        # est fautive : la presence des clefs ne suffit donc pas a disculper l'appel.
        if a == "Finish" and req and all(key_present(raw, k) for k in req):
            return None
        if not req:
            return None
        return "input not an object"
    missing = [p for p in req if p not in val]
    return "required argument missing" if missing else None

PATTERNS = [
    ("argument validation", re.compile(
        r"missing \d+ required positional argument|must have \"|unexpected keyword"
        r"|required (parameter|argument)|takes \d+ positional", re.I)),
    ("bad request (400)", re.compile(r"status_code=400")),
    ("auth or quota", re.compile(r"status_code=(401|403|429)|exceeded the \w+ quota", re.I)),
    ("not found (404)", re.compile(r"status_code=404")),
    ("service fault (5xx)", re.compile(r"status_code=5\d\d|Heroku", re.I)),
    ("tool-side exception", re.compile(r"Function executing from .* error", re.I)),
    ("timeout or connection", re.compile(r"timed? ?out|connection|unreachable", re.I)),
]
ANY_ERROR = re.compile(r'"error"\s*:\s*"(?!\s*")|error\s*:\s*"|status_code=(4|5)\d\d'
                       r'|missing \d+ required positional argument', re.I)

def classify(obs):
    s = str(obs)
    if not ANY_ERROR.search(s):
        return None
    for name, pat in PATTERNS:
        if pat.search(s):
            return name
    return "unclassified"

analysed = excluded = 0
kinds, rej = Counter(), Counter()
tp = fp = fn = tn = 0
tp_v = fn_v = 0            # contre la seule validation d'arguments
giveup = answered = 0
keep = []

for r in rows:
    sc = schemas(r)
    st = steps(r) if sc else []
    if not sc or not st:
        excluded += 1
        continue
    analysed += 1
    by = {t["name"]: t for t in sc}
    keep.append(st)
    for s in st:
        f = schema_flag(s, by)
        if f:
            kinds[f] += 1
        if s["observation"] is None:
            continue
        k = classify(s["observation"])
        if k:
            rej[k] += 1
        if f and k:
            tp += 1
        elif f and not k:
            fp += 1
        elif (not f) and k:
            fn += 1
        else:
            tn += 1
        if k == "argument validation":
            tp_v += bool(f)
            fn_v += not f
    for s in st:
        if s["action"] != "Finish":
            continue
        v, _ = parse_input(s["input_raw"])
        rt = (v or {}).get("return_type")
        if rt is None:
            m = re.search(r'["\']return_type["\']\s*:\s*["\'](\w+)', s["input_raw"])
            rt = m.group(1) if m else ""
        giveup += rt == "give_up_and_restart"
        answered += rt == "give_answer"

steps_total = sum(kinds.values()) + tp + fp + fn + tn
print("analysed %d records, excluded %d" % (analysed, excluded))
print("schema flags by kind:")
for k, v in kinds.most_common():
    print("   %-28s %5d" % (k, v))

In [ ]:
def mcc(tp, fp, fn, tn):
    d = math.sqrt((tp + fp) * (tp + fn) * (tn + fp) * (tn + fn))
    return (tp * tn - fp * fn) / d if d else float("nan")

print("the free schema check against the service's own rejection")
print("   tp %5d   fp %5d" % (tp, fp))
print("   fn %5d   tn %5d" % (fn, tn))
print("   precision %.3f | recall over every rejection %.3f | MCC %+.3f"
      % (tp / max(1, tp + fp), tp / max(1, tp + fn), mcc(tp, fp, fn, tn)))
print()
print("   on the fault class the check targets, argument validation:")
print("      anticipated %d of %d, recall %.3f"
      % (tp_v, tp_v + fn_v, tp_v / max(1, tp_v + fn_v)))
print()
print("rejections by kind")
tot = sum(rej.values())
for k, v in rej.most_common():
    print("   %-26s %6d  %5.1f%%" % (k, v, 100.0 * v / max(1, tot)))
print()
print("give-up share %.1f%% (%d give_up_and_restart, %d give_answer)"
      % (100.0 * giveup / max(1, giveup + answered), giveup, answered))

In [ ]:
# contradiction et duplication, avec la distribution qui situe le seuil
NUM = re.compile(r"\d[\d\s]{4,}")

def numbers(t):
    out = set()
    for m in NUM.finditer(str(t).replace(" ", "")):
        d = re.sub(r"\D", "", m.group())
        if d:
            out.add(int(d))
    return out

fires = 0
for st in keep:
    retrieved = set()
    for s in st[:-1]:
        if s["observation"] and not classify(s["observation"]):
            retrieved |= numbers(s["observation"])
    last = st[-1]
    if last["action"] != "Finish":
        continue
    for v in numbers(last["input_raw"]):
        if v > 50000 and v not in retrieved and not any(
                abs(v - (a + b)) < 1000 or abs(v - (a - b)) < 1000
                for a in retrieved for b in retrieved):
            fires += 1
            break
print("contradiction test fires on %d of %d records (%.2f%%)"
      % (fires, analysed, 100.0 * fires / max(1, analysed)))

from sentence_transformers import SentenceTransformer
import numpy as np
enc = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2", device="cpu")
sims = []
for st in keep[:N_EMB]:
    if len(st) < 2:
        continue
    e = enc.encode(["%s %s %s" % (s["thought"], s["action"], s["input_raw"]) for s in st],
                   normalize_embeddings=True)
    g = e @ e.T
    np.fill_diagonal(g, -1.0)
    sims.append(float(g.max()))
sims = np.array(sims)
print("duplicate test over %d trajectories with two or more steps" % len(sims))
print("   deciles %s" % " ".join("%.3f" % v for v in np.percentile(sims, range(10, 100, 10))))
for th in (0.85, 0.90, 0.92, 0.95):
    print("   above %.2f : %.1f%%" % (th, 100.0 * (sims > th).mean()))

In [ ]:
print("=== settled figures for the manuscript ===")
print("corpus            ToolBench released trajectories, seed %d" % SEED)
print("records           %d analysed, %d excluded" % (analysed, excluded))
print("steps scored      %d carrying an observation" % (tp + fp + fn + tn))
print("schema flags      %d" % sum(kinds.values()))
print("precision         %.3f  (%d of %d flagged calls the service also rejected)"
      % (tp / max(1, tp + fp), tp, tp + fp))
print("recall, targeted  %.3f  (%d of %d argument-validation rejections)"
      % (tp_v / max(1, tp_v + fn_v), tp_v, tp_v + fn_v))
print("infrastructure    %.1f%% of all rejections"
      % (100.0 * sum(v for k, v in rej.items()
                     if k in ("service fault (5xx)", "timeout or connection",
                              "auth or quota", "not found (404)", "unclassified"))
         / max(1, sum(rej.values()))))
print("give-up share     %.1f%%" % (100.0 * giveup / max(1, giveup + answered)))
print("contradiction     %.2f%% of records" % (100.0 * fires / max(1, analysed)))
print("duplicate at 0.92 %.1f%% of trajectories" % (100.0 * (sims > 0.92).mean()))